# Whisper Diarization Studio on Kaggle: the meeting & audio OS behind your Cloudflare Tunnel

Drop in a meeting recording, podcast, lecture or video and get back:
- a transcript with **who spoke when** (faster-whisper `large-v3-turbo` + pyannote speaker diarization);
- a **meeting summary** with decisions, action items and open questions (local Ollama model, no paid API);
- downloads as Markdown, plain text, SRT and VTT subtitles, or JSON with word timings.

Inputs: upload (≤100 MB through Cloudflare), any URL yt-dlp understands (direct files, YouTube, podcast pages), or files attached to the notebook as a Kaggle dataset (no size limit). Speakers can be renamed; the summary then uses the real names.

| Path | What |
|---|---|
| `/` | The studio page |
| `/api/*` | JSON API (jobs, uploads, downloads, speakers, summaries) |
| `/mcp` | MCP server for agents (tools: `transcribe`, `get_job`, `get_transcript`, `list_jobs`, `rename_speakers`, `summarize`, `delete_job`, `list_input_files`, `server_status`) |

### Before you run
1. **Accelerator:** GPU T4 x2 (Whisper on GPU 0; speaker diarization and the summary model on GPU 1). One GPU works too. **Internet:** on.
2. **A Cloudflare Tunnel** with a public hostname pointing to `http://localhost:7860`.
3. **Kaggle secrets** (Add-ons → Secrets, then tick them for this notebook):
   - `WHISPER_TUNNEL_TOKEN`: the tunnel token. Without it everything still runs, only without a public URL.
   - `WHISPER_UI_PASSWORD` (recommended): browser login is any username + this password; agents send `Authorization: Bearer <password>`. Without it a random password is printed below.
   - `HF_TOKEN` (for speaker labels): a Hugging Face read token whose account accepted the terms of [pyannote/speaker-diarization-3.1](https://huggingface.co/pyannote/speaker-diarization-3.1) and [pyannote/segmentation-3.0](https://huggingface.co/pyannote/segmentation-3.0). Without it you still get transcripts, just without speakers.
4. **Run All.** First run: about 8 minutes (installs, Whisper model, Ollama model). A one-hour meeting then takes very roughly 5–10 minutes on a T4.

Jobs are stored in `/kaggle/working/whisper_studio/jobs/` and are wiped when the session ends: download what you want to keep. Logs: `/kaggle/working/logs/`.

In [ ]:
# 1. Settings (edit if you like)
WHISPER_MODEL = "large-v3-turbo"                       # or "large-v3" (slower, a bit more accurate), "medium", "small"
DIARIZATION_MODEL = "pyannote/speaker-diarization-3.1"
SUMMARY_MODEL = "qwen2.5:7b"                           # any Ollama model; "" turns summaries off (saves ~5 GB download)
PORT = 7860
WORK_DIR = "/kaggle/working/whisper_studio"
LOG_DIR = "/kaggle/working/logs"
APP_DIR = '/kaggle/working/whisper_app'
print("Settings saved.")

### 2. App files
These cells only write files into `APP_DIR`.

In [ ]:
# Shared web kit: password, MCP endpoint, proxy, process helpers
# Writes studio_http.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/whisper_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'studio_http.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""studio_http: the small web kit behind the newer agentic-notebooks apps.

build_notebook.py copies this file into each notebook's app folder, so every notebook stays
self-contained: nothing is imported from GitHub at run time. Standard library only.

An app built on it gets:
- routes with regex paths, JSON in and out, file (with byte ranges), zip and chunked responses;
- one password for everything: browsers use HTTP Basic (any username), agents send
  ``Authorization: Bearer <password>`` or ``X-Access-Token: <password>``. ``GET /health`` stays open;
- an MCP server at ``POST /mcp`` (Streamable HTTP transport, plain JSON responses, no sessions),
  and the same tools as REST: ``GET /mcp/tools`` and ``POST /mcp/tools/<name>`` with the arguments as JSON;
- an optional reverse proxy (HTTP and WebSocket) for every path no route matches;
- IPv4 and IPv6 listeners on one port (cloudflared may dial [::1] for "localhost").

The notebook cells use the helpers at the bottom: kaggle_secret(), spawn() / stop_process() with pid
files (so re-running a cell, or the whole notebook after a kernel restart, never leaves a second copy
running), wait_http(), tail(), start_tunnel() and keep_alive(). Logs go to files, never to the notebook.
"""
import base64
import hmac
import http.client
import json
import logging
import mimetypes
import os
import re
import select
import signal
import socket
import subprocess
import sys
import threading
import time
import traceback
import urllib.error
import urllib.request
import zipfile
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, quote, urlparse

MCP_VERSIONS = ("2025-06-18", "2025-03-26", "2024-11-05")
HOP = {"connection", "keep-alive", "proxy-authenticate", "proxy-authorization", "te", "trailer", "trailers",
       "transfer-encoding", "upgrade"}
SECRET_HEADERS = {"authorization", "x-access-token"}
PID_DIR = "/kaggle/working/.pids" if os.path.isdir("/kaggle/working") else "/tmp/agentic-notebooks-pids"


# ====================================================================== responses
class HTTPError(Exception):
    def __init__(self, status, message):
        super().__init__(message)
        self.status = status


class Response:
    def __init__(self, body=b"", status=200, ctype="text/plain; charset=utf-8", headers=None):
        self.body = body.encode("utf-8") if isinstance(body, str) else body
        self.status, self.ctype, self.headers = status, ctype, dict(headers or {})


class FileResponse:
    """A file from disk. ``name`` makes it a download. Single byte ranges work (audio/video seeking)."""

    def __init__(self, path, name=None, ctype=None, cache="no-cache"):
        if not os.path.isfile(path):
            raise HTTPError(404, "file not found")
        self.path, self.name, self.cache = path, name, cache
        self.ctype = ctype or mimetypes.guess_type(path)[0] or "application/octet-stream"


class StreamResponse:
    """Chunked response from an iterable of bytes or str."""

    def __init__(self, chunks, ctype="text/plain; charset=utf-8", headers=None):
        self.chunks, self.ctype, self.headers = chunks, ctype, dict(headers or {})


class ZipResponse:
    """Streams ``folder`` (or a list of (path, arcname) pairs) as a zip download."""

    def __init__(self, files, name):
        if isinstance(files, str):
            root = os.path.dirname(os.path.abspath(files))
            pairs = []
            for dp, dn, fn in os.walk(files):
                dn.sort()
                for f in sorted(fn):
                    if not f.endswith((".tmp", ".part")):
                        pairs.append((os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), root)))
            files = pairs
        self.files, self.name = files, name


class ToolContent(list):
    """Return this from an MCP tool to send raw content blocks, for example an image."""


def text_block(text):
    return {"type": "text", "text": text}


def image_block(data, mime="image/png"):
    return {"type": "image", "data": base64.b64encode(data).decode("ascii"), "mimeType": mime}


def file_logger(path, name="app"):
    """A logger that writes to ``path`` only (nothing reaches the notebook output)."""
    log = logging.getLogger(name)
    path = os.path.abspath(path)
    if not any(getattr(h, "baseFilename", None) == path for h in log.handlers):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        h = logging.FileHandler(path, encoding="utf-8")
        h.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%Y-%m-%d %H:%M:%S"))
        log.addHandler(h)
    log.setLevel(logging.INFO)
    log.propagate = False
    return log


def _disposition(name):
    ascii_name = re.sub(r'[^A-Za-z0-9_.() -]+', "_", name) or "download"
    return "attachment; filename=\"%s\"; filename*=UTF-8''%s" % (ascii_name, quote(name))


# ====================================================================== requests
class Request:
    def __init__(self, handler, params, max_body):
        self.handler = handler
        u = urlparse(handler.path)
        self.method = handler.command
        self.path = u.path
        self.query = {k: v[-1] for k, v in parse_qs(u.query, keep_blank_values=True).items()}
        self.params = params
        self.headers = handler.headers
        self.max_body = max_body
        self.consumed = False

    @property
    def length(self):
        try:
            return int(self.headers.get("Content-Length") or 0)
        except ValueError:
            raise HTTPError(400, "bad Content-Length")

    @property
    def chunked(self):
        return "chunked" in (self.headers.get("Transfer-Encoding") or "").lower()

    def has_body(self):
        return self.chunked or self.length > 0

    def iter_body(self, limit=None):
        """Yields the request body in pieces (Content-Length or chunked), enforcing ``limit`` bytes."""
        limit = self.max_body if limit is None else limit
        too_big = HTTPError(413, "request body too large (limit %d MB)" % (limit // 2 ** 20))
        rf = self.handler.rfile
        self.consumed = True
        if self.chunked:
            total = 0
            while True:
                line = rf.readline(1024)
                try:
                    size = int(line.split(b";")[0].strip() or b"0", 16)
                except ValueError:
                    raise HTTPError(400, "bad chunked body")
                if size == 0:
                    while rf.readline(1024) not in (b"\r\n", b"\n", b""):
                        pass
                    return
                total += size
                if total > limit:
                    raise too_big
                left = size
                while left:
                    data = rf.read(min(left, 1 << 20))
                    if not data:
                        raise HTTPError(400, "request body was cut off")
                    left -= len(data)
                    yield data
                rf.readline(8)
        else:
            left = self.length
            if left > limit:
                raise too_big
            while left:
                data = rf.read(min(left, 1 << 20))
                if not data:
                    raise HTTPError(400, "request body was cut off")
                left -= len(data)
                yield data

    def body(self):
        return b"".join(self.iter_body())

    def json(self):
        raw = self.body()
        if not raw.strip():
            return {}
        try:
            return json.loads(raw.decode("utf-8"))
        except ValueError:
            raise HTTPError(400, "body is not valid JSON")

    def save_body(self, path, limit):
        """Writes the raw body to ``path`` (atomically) and returns its size."""
        if not self.has_body():
            raise HTTPError(411, "send the file as the raw request body")
        tmp = path + ".part"
        size = 0
        try:
            with open(tmp, "wb") as f:
                for data in self.iter_body(limit):
                    f.write(data)
                    size += len(data)
        except BaseException:
            if os.path.exists(tmp):
                os.remove(tmp)
            raise
        os.replace(tmp, path)
        return size

    def arg(self, name, default=None, cast=str):
        v = self.query.get(name)
        if v in (None, ""):
            return default
        try:
            return cast(v)
        except (TypeError, ValueError):
            raise HTTPError(400, "bad value for %s" % name)


# ====================================================================== app
def _rpc_error(mid, code, message):
    return {"jsonrpc": "2.0", "id": mid, "error": {"code": code, "message": message}}


class App:
    def __init__(self, name, password=None, version="1.0", instructions="", proxy=None, log=None,
                 max_body=64 << 20):
        self.name, self.version, self.instructions = name, version, instructions
        self.password = password or None
        self.proxy = proxy.rstrip("/") if proxy else None
        self.log = log or logging.getLogger(name)
        self.max_body = max_body
        self.routes, self.tools, self.servers = [], {}, []

        self.route("GET", "/health", auth=False)(lambda req: {"ok": True})
        self.route("POST", "/mcp")(self._mcp_http)
        self.route("GET", "/mcp")(lambda req: Response(
            "This MCP server answers POST requests (Streamable HTTP transport, JSON responses).", 405,
            headers={"Allow": "POST"}))
        self.route("DELETE", "/mcp")(lambda req: Response(b"", 405, headers={"Allow": "POST"}))
        self.route("GET", "/mcp/tools")(lambda req: {"tools": self.tool_list()})
        self.route("POST", r"/mcp/tools/(?P<name>[A-Za-z0-9_\-]+)")(self._tool_http)

    # ------------------------------------------------------------------ registration
    def route(self, method, pattern, auth=True):
        rx = re.compile(pattern)

        def deco(fn):
            self.routes.append((method.upper(), rx, fn, auth))
            return fn
        return deco

    def page(self, path, file):
        self.route("GET", re.escape(path))(lambda req: FileResponse(file, ctype="text/html; charset=utf-8"))

    def static(self, prefix, folder):
        root = os.path.realpath(folder)

        def serve(req):
            full = os.path.realpath(os.path.join(root, req.params["rel"]))
            if not full.startswith(root + os.sep):
                raise HTTPError(404, "not found")
            return FileResponse(full)
        self.route("GET", re.escape(prefix) + r"(?P<rel>.+)")(serve)

    def tool(self, name, description, properties=None, required=()):
        """Registers an MCP tool. ``properties`` is a JSON Schema properties object."""
        def deco(fn):
            self.tools[name] = {"fn": fn, "schema": {
                "name": name, "description": description,
                "inputSchema": {"type": "object", "properties": properties or {}, "required": list(required)}}}
            return fn
        return deco

    def tool_list(self):
        return [t["schema"] for t in self.tools.values()]

    def call_tool(self, name, args):
        """Runs a tool. Returns (content_blocks, is_error)."""
        t = self.tools[name]
        if not isinstance(args, dict):
            return [text_block("arguments must be a JSON object")], True
        try:
            out = t["fn"](**args)
        except TypeError as e:
            if "argument" in str(e):
                return [text_block("bad arguments for %s: %s" % (name, e))], True
            self.log.error("tool %s failed\n%s", name, traceback.format_exc())
            return [text_block("TypeError: %s" % e)], True
        except HTTPError as e:
            return [text_block(str(e))], True
        except (ValueError, KeyError) as e:
            return [text_block(str(e).strip("'\""))], True
        except Exception as e:
            self.log.error("tool %s failed\n%s", name, traceback.format_exc())
            return [text_block("%s: %s" % (type(e).__name__, e))], True
        if isinstance(out, ToolContent):
            return list(out), False
        if isinstance(out, str):
            return [text_block(out)], False
        return [text_block(json.dumps(out, indent=1, ensure_ascii=False, default=str))], False

    # ------------------------------------------------------------------ MCP
    def _mcp_http(self, req):
        try:
            msg = json.loads(req.body().decode("utf-8") or "null")
        except ValueError:
            return _rpc_error(None, -32700, "parse error")
        if isinstance(msg, list):
            out = [r for r in map(self._rpc, msg) if r is not None]
            return out if out else Response(b"", 202)
        r = self._rpc(msg)
        return r if r is not None else Response(b"", 202)

    def _rpc(self, m):
        if not isinstance(m, dict) or m.get("jsonrpc") != "2.0":
            return _rpc_error(m.get("id") if isinstance(m, dict) else None, -32600, "invalid request")
        if "method" not in m or "id" not in m:
            return None            # a notification, or a response to a request we never send
        mid, method, params = m["id"], m["method"], m.get("params") or {}
        try:
            if method == "initialize":
                v = params.get("protocolVersion")
                result = {"protocolVersion": v if v in MCP_VERSIONS else MCP_VERSIONS[0],
                          "capabilities": {"tools": {"listChanged": False}},
                          "serverInfo": {"name": self.name, "version": self.version}}
                if self.instructions:
                    result["instructions"] = self.instructions
            elif method == "ping":
                result = {}
            elif method == "tools/list":
                result = {"tools": self.tool_list()}
            elif method == "tools/call":
                name = params.get("name")
                if name not in self.tools:
                    return _rpc_error(mid, -32602, "unknown tool: %s" % name)
                content, err = self.call_tool(name, params.get("arguments") or {})
                result = {"content": content, "isError": err}
            elif method == "resources/list":
                result = {"resources": []}
            elif method == "resources/templates/list":
                result = {"resourceTemplates": []}
            elif method == "prompts/list":
                result = {"prompts": []}
            else:
                return _rpc_error(mid, -32601, "method not found: %s" % method)
        except Exception as e:
            self.log.error("mcp %s failed\n%s", method, traceback.format_exc())
            return _rpc_error(mid, -32603, str(e))
        return {"jsonrpc": "2.0", "id": mid, "result": result}

    def _tool_http(self, req):
        name = req.params["name"]
        if name not in self.tools:
            raise HTTPError(404, "unknown tool: %s" % name)
        content, err = self.call_tool(name, req.json())
        if len(content) == 1 and content[0]["type"] == "text":
            try:
                data = json.loads(content[0]["text"])
            except ValueError:
                data = content[0]["text"]
            body = {"error": data} if err else data
        else:
            body = {"content": content, "isError": err}
        return Response(json.dumps(body, ensure_ascii=False, default=str), 400 if err else 200,
                        "application/json; charset=utf-8")

    # ------------------------------------------------------------------ auth
    def authorized(self, headers):
        pw = self.password
        if not pw:
            return True

        def ok(given):
            return bool(given) and hmac.compare_digest(given.encode("utf-8"), pw.encode("utf-8"))
        if ok(headers.get("X-Access-Token", "")):
            return True
        h = headers.get("Authorization", "")
        if h[:6].lower() == "basic ":
            try:
                given = base64.b64decode(h[6:].strip()).decode("utf-8").partition(":")[2]
            except Exception:
                given = ""
            return ok(given)
        if h[:7].lower() == "bearer ":
            return ok(h[7:].strip())
        return False

    def _deny(self, h):
        self._respond(h, Response(json.dumps({"error": "login required: HTTP Basic (any username) or "
                                              "'Authorization: Bearer <password>'"}), 401,
                                  "application/json; charset=utf-8",
                                  {"WWW-Authenticate": 'Basic realm="%s", charset="UTF-8"' % self.name}))

    # ------------------------------------------------------------------ dispatch
    def dispatch(self, h):
        h._sent = False
        req = None
        try:
            path = urlparse(h.path).path
            method = "GET" if h.command == "HEAD" else h.command
            found, path_known = None, False
            for m, rx, fn, auth in self.routes:
                mt = rx.fullmatch(path)
                if mt:
                    path_known = True
                    if m == method:
                        found = (fn, auth, mt.groupdict())
                        break
            if found is None:
                if path_known:
                    raise HTTPError(405, "method not allowed")
                if not self.proxy:
                    raise HTTPError(404, "not found")
                if not self.authorized(h.headers):
                    return self._deny(h)
                return self._proxy(h)
            fn, auth, params = found
            if auth and not self.authorized(h.headers):
                return self._deny(h)
            req = Request(h, params, self.max_body)
            self._respond(h, fn(req))
        except (BrokenPipeError, ConnectionResetError):
            h.close_connection = True
        except HTTPError as e:
            self._fail(h, e.status, str(e))
        except (ValueError, KeyError) as e:
            msg = ("missing field %s" % e) if isinstance(e, KeyError) else str(e)
            self._fail(h, 400, msg)
        except Exception as e:
            self.log.error("%s %s failed\n%s", h.command, h.path, traceback.format_exc())
            self._fail(h, 500, "%s: %s" % (type(e).__name__, e))
        finally:
            body_left = (req is None or not req.consumed) and (
                "chunked" in (h.headers.get("Transfer-Encoding") or "").lower()
                or (h.headers.get("Content-Length") or "0").strip() not in ("", "0"))
            if body_left:
                h.close_connection = True   # an unread body would corrupt the next request on this connection

    def _fail(self, h, status, message):
        if h._sent:
            h.close_connection = True
            return
        try:
            self._respond(h, Response(json.dumps({"error": message}), status, "application/json; charset=utf-8"))
        except OSError:
            h.close_connection = True

    def _start(self, h, status, headers):
        h.send_response(status)
        h._sent = True
        for k, v in headers:
            h.send_header(k, v)
        h.end_headers()

    def _respond(self, h, out):
        head = h.command == "HEAD"
        if out is None:
            out = Response(b"", 204)
        elif isinstance(out, (dict, list)):
            out = Response(json.dumps(out, ensure_ascii=False, default=str), 200, "application/json; charset=utf-8")
        elif isinstance(out, str):
            out = Response(out)
        elif isinstance(out, bytes):
            out = Response(out, 200, "application/octet-stream")

        if isinstance(out, Response):
            hdrs = {"Content-Type": out.ctype, "Cache-Control": "no-store"}
            hdrs.update(out.headers)
            if out.status not in (204, 304):
                hdrs["Content-Length"] = str(len(out.body))
            self._start(h, out.status, hdrs.items())
            if not head and out.status not in (204, 304) and out.body:
                h.wfile.write(out.body)
        elif isinstance(out, FileResponse):
            self._send_file(h, out, head)
        elif isinstance(out, StreamResponse):
            self._start(h, 200, list({"Content-Type": out.ctype, "Cache-Control": "no-cache",
                                      "X-Accel-Buffering": "no", **out.headers}.items())
                        + [("Transfer-Encoding", "chunked")])
            if head:
                h.wfile.write(b"0\r\n\r\n")
                return
            for chunk in out.chunks:
                if isinstance(chunk, str):
                    chunk = chunk.encode("utf-8")
                if chunk:
                    h.wfile.write(b"%x\r\n" % len(chunk) + chunk + b"\r\n")
                    h.wfile.flush()
            h.wfile.write(b"0\r\n\r\n")
        elif isinstance(out, ZipResponse):
            self._send_zip(h, out, head)
        else:
            raise TypeError("handler returned %r" % type(out))

    def _send_file(self, h, f, head):
        size = os.path.getsize(f.path)
        start, end, status = 0, size - 1, 200
        m = re.fullmatch(r"bytes=(\d*)-(\d*)", (h.headers.get("Range") or "").strip())
        if m and size and (m.group(1) or m.group(2)):
            a, b = m.groups()
            if a == "":
                start = max(0, size - int(b))
            else:
                start, end = int(a), min(int(b), size - 1) if b else size - 1
            if start > end or start >= size:
                return self._start(h, 416, [("Content-Range", "bytes */%d" % size), ("Content-Length", "0")])
            status = 206
        hdrs = [("Content-Type", f.ctype), ("Content-Length", str(end - start + 1)), ("Accept-Ranges", "bytes"),
                ("Cache-Control", f.cache)]
        if status == 206:
            hdrs.append(("Content-Range", "bytes %d-%d/%d" % (start, end, size)))
        if f.name:
            hdrs.append(("Content-Disposition", _disposition(f.name)))
        self._start(h, status, hdrs)
        if head:
            return
        with open(f.path, "rb") as fh:
            fh.seek(start)
            left = end - start + 1
            while left > 0:
                data = fh.read(min(left, 1 << 20))
                if not data:
                    break
                h.wfile.write(data)
                left -= len(data)

    def _send_zip(self, h, z, head):
        self._start(h, 200, [("Content-Type", "application/zip"), ("Content-Disposition", _disposition(z.name)),
                             ("Cache-Control", "no-store"), ("Transfer-Encoding", "chunked")])
        w = h.wfile

        class Chunks:
            def write(self, data):
                if data:
                    w.write(b"%x\r\n" % len(data) + bytes(data) + b"\r\n")
                return len(data)

            def flush(self):
                w.flush()

        if not head:
            with zipfile.ZipFile(Chunks(), "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
                for path, arc in z.files:
                    if isinstance(path, bytes):
                        zf.writestr(arc, path)
                    elif os.path.isfile(path):
                        zf.write(path, arc, compress_type=zipfile.ZIP_STORED if path.endswith(
                            (".safetensors", ".gguf", ".bin", ".npy", ".png", ".jpg", ".mp4", ".zip", ".wav"))
                            else zipfile.ZIP_DEFLATED)
        w.write(b"0\r\n\r\n")
        w.flush()

    # ------------------------------------------------------------------ reverse proxy
    def _proxy(self, h):
        pu = urlparse(self.proxy)
        host, port = pu.hostname, pu.port or 80
        if ("upgrade" in (h.headers.get("Connection") or "").lower()
                and (h.headers.get("Upgrade") or "").lower() == "websocket"):
            return self._proxy_ws(h, host, port)
        req = Request(h, {}, self.max_body)
        headers = [(k, v) for k, v in h.headers.items()
                   if k.lower() not in HOP and k.lower() not in SECRET_HEADERS and k.lower() != "content-length"]
        headers.append(("X-Forwarded-For", h.client_address[0]))
        body = req.body() if req.has_body() else None
        if body is not None:
            headers.append(("Content-Length", str(len(body))))
        conn = http.client.HTTPConnection(host, port, timeout=900)
        try:
            # the original Host header is kept: some backends (ComfyUI) compare it with Origin
            conn.putrequest(h.command, h.path, skip_host="host" in {k.lower() for k, v in headers},
                            skip_accept_encoding=True)
            for k, v in headers:
                conn.putheader(k, v)
            conn.endheaders(body)
            r = conn.getresponse()
        except OSError as e:
            conn.close()
            raise HTTPError(502, "backend not reachable (%s)" % type(e).__name__)
        try:
            hdrs = [(k, v) for k, v in r.getheaders() if k.lower() not in HOP and k.lower() != "content-length"]
            cl = r.getheader("Content-Length")
            chunked = "chunked" in (r.getheader("Transfer-Encoding") or "").lower()
            if h.command == "HEAD" or r.status in (204, 304) or r.status < 200:
                return self._start(h, r.status, hdrs + ([("Content-Length", cl)] if cl else []))
            if cl is not None and not chunked:
                self._start(h, r.status, hdrs + [("Content-Length", cl)])
                while True:
                    data = r.read(1 << 16)
                    if not data:
                        break
                    h.wfile.write(data)
            else:
                self._start(h, r.status, hdrs + [("Transfer-Encoding", "chunked")])
                while True:
                    data = r.read1(1 << 16)
                    if not data:
                        break
                    h.wfile.write(b"%x\r\n" % len(data) + data + b"\r\n")
                    h.wfile.flush()
                h.wfile.write(b"0\r\n\r\n")
        finally:
            conn.close()

    def _proxy_ws(self, h, host, port):
        try:
            up = socket.create_connection((host, port), timeout=10)
        except OSError as e:
            raise HTTPError(502, "backend not reachable (%s)" % type(e).__name__)
        lines = ["%s %s HTTP/1.1" % (h.command, h.path)]
        lines += ["%s: %s" % (k, v) for k, v in h.headers.items() if k.lower() not in SECRET_HEADERS]
        h._sent = True
        h.close_connection = True
        client = h.connection
        try:
            up.sendall(("\r\n".join(lines) + "\r\n\r\n").encode("latin-1"))
            up.settimeout(None)
            client.settimeout(None)
            while True:
                ready, _, broken = select.select([client, up], [], [client, up], 600)
                if broken:
                    break
                for s in ready:
                    data = s.recv(1 << 16)
                    if not data:
                        return
                    (up if s is client else client).sendall(data)
        except OSError:
            pass
        finally:
            up.close()

    # ------------------------------------------------------------------ serving
    def handler_class(self):
        app = self

        class Handler(BaseHTTPRequestHandler):
            protocol_version = "HTTP/1.1"
            server_version = re.sub(r"\W+", "", app.name) or "studio"

            def log_message(self, fmt, *args):
                pass

            def do_GET(self):
                app.dispatch(self)

            do_POST = do_PUT = do_PATCH = do_DELETE = do_HEAD = do_OPTIONS = do_GET
        return Handler

    def start(self, port):
        """Listens on 0.0.0.0 and [::] in background threads. Calling it again restarts cleanly."""
        self.stop()
        handler = self.handler_class()
        for cls, host in ((_V4Server, "0.0.0.0"), (_V6Server, "::")):
            try:
                srv = cls((host, port), handler)
            except OSError as e:
                self.log.warning("could not listen on [%s]:%d: %s", host, port, e)
                continue
            threading.Thread(target=srv.serve_forever, daemon=True).start()
            self.servers.append(srv)
        if not self.servers:
            raise RuntimeError("nothing is listening on port %d" % port)
        self.log.info("%s listening on port %d", self.name, port)

    def stop(self):
        while self.servers:
            srv = self.servers.pop()
            try:
                srv.shutdown()
                srv.server_close()
            except Exception:
                pass

    def serve_forever(self, port):
        """For app processes started with spawn(): serve until SIGTERM."""
        done = threading.Event()
        signal.signal(signal.SIGTERM, lambda *a: done.set())
        signal.signal(signal.SIGINT, lambda *a: done.set())
        self.start(port)
        done.wait()
        self.log.info("stopping")
        self.stop()


class _V4Server(ThreadingHTTPServer):
    allow_reuse_address = True
    daemon_threads = True
    request_queue_size = 64


class _V6Server(_V4Server):
    address_family = socket.AF_INET6

    def server_bind(self):
        self.socket.setsockopt(socket.IPPROTO_IPV6, socket.IPV6_V6ONLY, 1)
        super().server_bind()


# ====================================================================== JSON files and jobs
def read_json(path, default=None):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default


def write_json(path, data):
    """Atomic write, so readers in other threads or processes never see half a file."""
    tmp = "%s.%d.%d.tmp" % (path, os.getpid(), threading.get_ident())
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=1, ensure_ascii=False, default=str)
    os.replace(tmp, path)


def safe_name(s, n=60):
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(s or "")).strip("-.")[:n]


def gpu_info():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5).stdout
        gpus = []
        for line in out.strip().splitlines():
            i, n, u, t, g = [x.strip() for x in line.split(",")]
            gpus.append({"index": int(i), "name": n, "mem_used_mb": int(u), "mem_total_mb": int(t), "util": int(g)})
        return gpus
    except Exception:
        return []


# ====================================================================== notebook helpers
def kaggle_secret(name):
    """Value of a Kaggle secret, or None when it is missing or not attached to this notebook.
    Outside Kaggle (local runs, Docker) it reads the environment variable of the same name."""
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return os.environ.get(name) or None
    try:
        return UserSecretsClient().get_secret(name) or None
    except Exception:
        return None


def ui_password(secret_name):
    pw = kaggle_secret(secret_name)
    if pw:
        print("UI password: from the %s secret. Browser login: any username + that password." % secret_name)
        return pw
    import secrets
    pw = secrets.token_urlsafe(9)
    print("No %s secret, so this session's password is: %s" % (secret_name, pw))
    print("Browser login: any username + that password. Add the secret to keep a fixed password.")
    return pw


def _pidfile(name):
    return os.path.join(PID_DIR, safe_name(name) + ".json")


def _alive(pid, exe=None):
    try:
        with open("/proc/%d/stat" % pid) as f:
            if f.read().rsplit(")", 1)[1].split()[0] == "Z":
                return False
        if exe:
            with open("/proc/%d/cmdline" % pid, "rb") as f:
                return exe.encode() in f.read()
        return True
    except (OSError, IndexError):
        return False


def process_alive(name):
    info = read_json(_pidfile(name))
    return bool(info) and _alive(info["pid"], info.get("exe"))


def stop_process(name, timeout=20):
    """Stops a process spawn() started under ``name`` (and its children). Returns True if one was running."""
    info = read_json(_pidfile(name))
    if not info:
        return False
    pid, exe = info["pid"], info.get("exe")
    running = _alive(pid, exe)
    if running:
        for sig, wait in ((signal.SIGTERM, timeout), (signal.SIGKILL, 5)):
            try:
                os.killpg(pid, sig)
            except OSError:
                break
            t = time.time()
            while time.time() - t < wait and _alive(pid, exe):
                time.sleep(0.2)
            if not _alive(pid, exe):
                break
    try:
        os.remove(_pidfile(name))
    except OSError:
        pass
    return running


def spawn(name, cmd, log_path, env=None, cwd=None):
    """Starts ``cmd`` in the background with its output appended to ``log_path``.
    A process started earlier under the same name is stopped first, so re-running a cell is safe."""
    stop_process(name)
    os.makedirs(PID_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(os.path.abspath(log_path)), exist_ok=True)
    with open(log_path, "ab") as log:
        log.write(("\n===== %s: starting %s =====\n" % (time.strftime("%Y-%m-%d %H:%M:%S"), name)).encode())
        log.flush()
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                                env=env, cwd=cwd, start_new_session=True)
    write_json(_pidfile(name), {"pid": proc.pid, "exe": os.path.basename(cmd[0]), "log": log_path,
                                "started": time.time()})
    return proc


def wait_http(url, timeout=120, name=None, headers=None):
    """Waits until ``url`` answers (any HTTP status). Returns False on timeout or if process ``name`` died."""
    t = time.time()
    while time.time() - t < timeout:
        try:
            urllib.request.urlopen(urllib.request.Request(url, headers=headers or {}), timeout=3)
            return True
        except urllib.error.HTTPError:
            return True
        except Exception:
            pass
        if name and not process_alive(name):
            return False
        time.sleep(1)
    return False


def tail(path, n=30):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 64 * 1024))
            return "\n".join(f.read().decode("utf-8", "replace").splitlines()[-n:])
    except OSError:
        return "(no log yet at %s)" % path


def ensure_cloudflared(path="/kaggle/working/cloudflared"):
    if not os.path.exists(path):
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
        os.chmod(path, 0o755)
    return path


def start_tunnel(secret_name, port=7860, log_path="/kaggle/working/logs/cloudflared.log",
                 binary="/kaggle/working/cloudflared"):
    """Starts cloudflared with the token from ``secret_name`` (passed as TUNNEL_TOKEN, never printed).
    A missing secret is not an error: the app keeps running inside the session, without a public URL."""
    stop_process("cloudflared")
    token = kaggle_secret(secret_name)
    if not token:
        print("No public URL: there is no Kaggle secret named %s attached to this notebook.\n"
              "The app is running inside this session anyway. To publish it:\n"
              "  1. Cloudflare Zero Trust -> Networks -> Tunnels: create a tunnel and add a public hostname\n"
              "     pointing to http://localhost:%d\n"
              "  2. Kaggle: Add-ons -> Secrets -> add %s with the tunnel token, and tick it for this notebook\n"
              "  3. Re-run this cell." % (secret_name, port, secret_name))
        return None
    ensure_cloudflared(binary)
    offset = os.path.getsize(log_path) if os.path.exists(log_path) else 0
    proc = spawn("cloudflared", [binary, "tunnel", "--no-autoupdate", "run"], log_path,
                 env=dict(os.environ, TUNNEL_TOKEN=token))

    def new_log():
        with open(log_path, "rb") as f:
            f.seek(offset)
            return f.read().decode("utf-8", "replace")
    for _ in range(45):
        time.sleep(1)
        if "Registered tunnel connection" in new_log():
            print("Tunnel connected. Open the public hostname you routed to http://localhost:%d" % port)
            return proc
        if proc.poll() is not None:
            print("cloudflared stopped (exit code %s). Last log lines:" % proc.returncode)
            print("\n".join(new_log().splitlines()[-15:]))
            print("Check that %s holds the token of a tunnel that still exists." % secret_name)
            return None
    print("Tunnel not registered yet; it keeps trying in the background. Last log lines:")
    print("\n".join(new_log().splitlines()[-15:]))
    return proc


def keep_alive(port, names, extra=None, interval=60):
    """The last cell: prints one status line a minute so the Kaggle session stays active.
    Interrupting it only stops the printing; the app keeps running."""
    try:
        while True:
            try:
                urllib.request.urlopen("http://127.0.0.1:%d/health" % port, timeout=5)
                parts = ["ui=up"]
            except Exception:
                parts = ["ui=DOWN"]
            for n in names:
                if n == "cloudflared" and not os.path.exists(_pidfile(n)):
                    parts.append("tunnel=off")
                else:
                    parts.append("%s=%s" % (n, "up" if process_alive(n) else "DOWN"))
            gpus = gpu_info()
            if gpus:
                parts.append("GPU " + ", ".join("%d: %.1f/%.0f GB %d%%" % (g["index"], g["mem_used_mb"] / 1024,
                                                                           g["mem_total_mb"] / 1024, g["util"])
                                                for g in gpus))
            if extra:
                try:
                    s = extra()
                    if s:
                        parts.append(s)
                except Exception as e:
                    parts.append("status error: %s" % e)
            print(time.strftime("%H:%M:%S"), " ".join(parts), flush=True)
            time.sleep(interval)
    except KeyboardInterrupt:
        print("Stopped watching. Everything keeps running in the background; re-run this cell to watch again.")


def api_get(port, path, password, timeout=10):
    """GET a JSON endpoint of a local app (used by keep-alive status lines)."""
    req = urllib.request.Request("http://127.0.0.1:%d%s" % (port, path),
                                 headers={"Authorization": "Bearer %s" % password} if password else {})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))
''')
print("wrote", os.path.join(APP_DIR, 'studio_http.py'))

In [ ]:
# Shared page styles
# Writes kit.css into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/whisper_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'kit.css'), "w", encoding="utf-8") as f:
    f.write(r'''/* Shared look for the agentic-notebooks app pages (served at /static/kit.css). */
:root{--bg:#f5f6f8;--panel:#fff;--panel2:#f0f2f5;--text:#1b1f24;--muted:#626b76;--line:#dde1e6;
  --accent:#2f6fec;--accent-ink:#fff;--ok:#1a7f4b;--warn:#a96500;--bad:#c4372f;--radius:10px;
  --mono:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace}
@media (prefers-color-scheme:dark){:root{--bg:#0f1114;--panel:#171a1f;--panel2:#1e2228;--text:#e7e9ec;
  --muted:#9aa3ad;--line:#2b3038;--accent:#5b8ff5;--ok:#3fbf7f;--warn:#e0a03a;--bad:#ef6a60}}
*{box-sizing:border-box}
html,body{margin:0;background:var(--bg);color:var(--text);font:14px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
a{color:var(--accent)}
header.top{display:flex;align-items:center;gap:16px;flex-wrap:wrap;padding:12px 20px;border-bottom:1px solid var(--line);background:var(--panel)}
header.top h1{font-size:17px;margin:0;font-weight:650}
header.top .sub{color:var(--muted);font-size:13px}
header.top .right{margin-left:auto;display:flex;gap:10px;align-items:center;flex-wrap:wrap}
nav.tabs{display:flex;gap:4px;padding:0 20px;border-bottom:1px solid var(--line);background:var(--panel);overflow-x:auto}
nav.tabs button{border:0;background:none;color:var(--muted);padding:10px 12px;font:inherit;cursor:pointer;border-bottom:2px solid transparent;white-space:nowrap}
nav.tabs button.on{color:var(--text);border-bottom-color:var(--accent);font-weight:600}
main{max-width:1180px;margin:0 auto;padding:20px}
[data-pane]{display:none}[data-pane].on{display:block}
.card{background:var(--panel);border:1px solid var(--line);border-radius:var(--radius);padding:16px;margin-bottom:16px}
.card h2{font-size:15px;margin:0 0 12px}
.card h3{font-size:14px;margin:14px 0 8px}
.grid{display:grid;gap:16px;grid-template-columns:repeat(auto-fit,minmax(280px,1fr))}
.row{display:flex;gap:10px;align-items:center;flex-wrap:wrap}
.col{display:flex;flex-direction:column;gap:6px}
.muted{color:var(--muted)}.small{font-size:12px}.mono{font-family:var(--mono);font-size:12.5px}
label{display:flex;flex-direction:column;gap:4px;font-size:12.5px;color:var(--muted)}
label.inline{flex-direction:row;align-items:center;gap:6px;color:var(--text);font-size:13px}
input,select,textarea{font:inherit;color:var(--text);background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:7px 9px;min-width:0}
input[type=checkbox]{width:auto}
textarea{width:100%;resize:vertical;min-height:90px}
input:focus,select:focus,textarea:focus{outline:2px solid color-mix(in srgb,var(--accent) 40%,transparent);border-color:var(--accent)}
button,.btn{font:inherit;border:1px solid var(--line);background:var(--panel2);color:var(--text);border-radius:8px;padding:7px 13px;cursor:pointer;text-decoration:none;display:inline-flex;align-items:center;gap:6px}
button.primary,.btn.primary{background:var(--accent);border-color:var(--accent);color:var(--accent-ink);font-weight:600}
button.danger{color:var(--bad)}
button:disabled{opacity:.5;cursor:default}
table{width:100%;border-collapse:collapse}
th,td{text-align:left;padding:7px 8px;border-bottom:1px solid var(--line);vertical-align:top}
th{font-size:12px;color:var(--muted);font-weight:600}
.badge{display:inline-block;padding:1px 8px;border-radius:999px;font-size:12px;background:var(--panel2);border:1px solid var(--line)}
.badge.ok{color:var(--ok)}.badge.warn{color:var(--warn)}.badge.bad{color:var(--bad)}
.bar{height:6px;background:var(--panel2);border-radius:99px;overflow:hidden}
.bar>i{display:block;height:100%;background:var(--accent);transition:width .3s}
pre.log{background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:10px;max-height:320px;overflow:auto;font:12px/1.45 var(--mono);white-space:pre-wrap;margin:0}
.drop{border:2px dashed var(--line);border-radius:var(--radius);padding:22px;text-align:center;color:var(--muted)}
.drop.over{border-color:var(--accent);color:var(--text)}
#toasts{position:fixed;right:16px;bottom:16px;display:flex;flex-direction:column;gap:8px;z-index:50}
.toast{background:var(--panel);border:1px solid var(--line);border-left:4px solid var(--accent);padding:10px 14px;border-radius:8px;box-shadow:0 6px 20px rgba(0,0,0,.15);max-width:380px}
.toast.bad{border-left-color:var(--bad)}.toast.ok{border-left-color:var(--ok)}
.empty{color:var(--muted);padding:18px;text-align:center}
@media (max-width:640px){main{padding:12px}header.top{padding:10px 12px}nav.tabs{padding:0 8px}}
''')
print("wrote", os.path.join(APP_DIR, 'kit.css'))

In [ ]:
# Shared page scripts
# Writes kit.js into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/whisper_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'kit.js'), "w", encoding="utf-8") as f:
    f.write(r'''// Shared helpers for the agentic-notebooks app pages (served at /static/kit.js).
// The browser already holds the HTTP Basic login, so fetch() calls need no token.
const $ = (sel, root = document) => root.querySelector(sel);
const $$ = (sel, root = document) => [...root.querySelectorAll(sel)];

function el(tag, attrs = {}, ...kids) {
  const e = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs || {})) {
    if (v == null || v === false) continue;
    if (k.startsWith("on")) e.addEventListener(k.slice(2), v);
    else if (k === "html") e.innerHTML = v;
    else if (k === "style" && typeof v === "object") Object.assign(e.style, v);
    else e.setAttribute(k, v === true ? "" : v);
  }
  for (const kid of kids.flat()) if (kid != null && kid !== false) e.append(kid.nodeType ? kid : String(kid));
  return e;
}

async function api(path, opts = {}) {
  const init = { method: opts.method || (opts.json !== undefined || opts.body ? "POST" : "GET"), headers: {} };
  if (opts.json !== undefined) { init.body = JSON.stringify(opts.json); init.headers["Content-Type"] = "application/json"; }
  else if (opts.body) init.body = opts.body;
  const r = await fetch(path, init);
  const ct = r.headers.get("Content-Type") || "";
  const data = ct.includes("json") ? await r.json() : await r.text();
  if (!r.ok) throw new Error((data && data.error) || r.status + " " + r.statusText);
  return data;
}

function toast(msg, kind = "") {
  let box = $("#toasts");
  if (!box) { box = el("div", { id: "toasts" }); document.body.append(box); }
  const t = el("div", { class: "toast " + kind }, msg);
  box.append(t);
  setTimeout(() => t.remove(), kind === "bad" ? 9000 : 4500);
}

async function guard(fn, okMsg) {
  try { const r = await fn(); if (okMsg) toast(okMsg, "ok"); return r; }
  catch (e) { toast(e.message || String(e), "bad"); throw e; }
}

function setupTabs(onChange) {
  const btns = $$("nav.tabs button[data-tab]");
  const show = (name) => {
    btns.forEach(b => b.classList.toggle("on", b.dataset.tab === name));
    $$("[data-pane]").forEach(p => p.classList.toggle("on", p.dataset.pane === name));
    try { history.replaceState(null, "", "#" + name); } catch (e) {}
    if (onChange) onChange(name);
  };
  btns.forEach(b => b.addEventListener("click", () => show(b.dataset.tab)));
  const first = location.hash.slice(1);
  show(btns.some(b => b.dataset.tab === first) ? first : btns[0].dataset.tab);
  return show;
}

function fmtBytes(n) {
  if (n == null) return "";
  const u = ["B", "KB", "MB", "GB", "TB"]; let i = 0;
  while (n >= 1024 && i < u.length - 1) { n /= 1024; i++; }
  return (i ? n.toFixed(1) : n) + " " + u[i];
}

function fmtDur(s) {
  if (s == null || isNaN(s)) return "";
  s = Math.max(0, Math.round(s));
  const h = Math.floor(s / 3600), m = Math.floor(s % 3600 / 60), sec = s % 60;
  return (h ? h + ":" + String(m).padStart(2, "0") : m) + ":" + String(sec).padStart(2, "0");
}

function fmtAgo(ts) {
  if (!ts) return "";
  const d = Date.now() / 1000 - ts;
  if (d < 60) return "just now";
  if (d < 3600) return Math.floor(d / 60) + " min ago";
  if (d < 86400) return Math.floor(d / 3600) + " h ago";
  return new Date(ts * 1000).toLocaleString();
}

function badge(state) {
  const kind = { done: "ok", ready: "ok", running: "warn", queued: "", loading: "warn", error: "bad", stopped: "bad" }[state] || "";
  return el("span", { class: "badge " + kind }, state || "?");
}

function progressBar(frac) {
  return el("div", { class: "bar" }, el("i", { style: { width: Math.round(Math.max(0, Math.min(1, frac || 0)) * 100) + "%" } }));
}

function every(ms, fn) {
  let stopped = false;
  const tick = async () => { if (stopped) return; try { await fn(); } catch (e) {} if (!stopped) setTimeout(tick, ms); };
  tick();
  return () => { stopped = true; };
}

function dropZone(zone, input, onFiles) {
  zone.addEventListener("click", () => input.click());
  input.addEventListener("change", () => { if (input.files.length) onFiles([...input.files]); input.value = ""; });
  zone.addEventListener("dragover", e => { e.preventDefault(); zone.classList.add("over"); });
  zone.addEventListener("dragleave", () => zone.classList.remove("over"));
  zone.addEventListener("drop", e => { e.preventDefault(); zone.classList.remove("over"); if (e.dataTransfer.files.length) onFiles([...e.dataTransfer.files]); });
}

function mcpHelp(name) {
  const url = location.origin + "/mcp";
  return "claude mcp add --transport http " + name + " " + url + " --header \"Authorization: Bearer <password>\"";
}
''')
print("wrote", os.path.join(APP_DIR, 'kit.js'))

In [ ]:
# Speaker assignment, transcript formats and summaries (pure Python)
# Writes whisper_core.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/whisper_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'whisper_core.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Pure-Python parts of Whisper Diarization Studio: speaker assignment, transcript formats and
meeting summaries. No GPU libraries are imported here, so this file is easy to test anywhere."""
import bisect
import json
import re
import urllib.request

FORMATS = {"txt": "text/plain", "md": "text/markdown", "srt": "application/x-subrip", "vtt": "text/vtt",
           "json": "application/json"}


# ====================================================================== timestamps
def ts(sec, sep=".", hours=True):
    sec = max(0.0, float(sec or 0))
    ms = int(round(sec * 1000))
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    s, ms = divmod(ms, 1000)
    return ("%02d:%02d:%02d%s%03d" % (h, m, s, sep, ms)) if hours else ("%02d:%02d" % (h * 60 + m, s))


def clock(sec):
    sec = int(max(0, sec or 0))
    return "%d:%02d:%02d" % (sec // 3600, sec // 60 % 60, sec % 60) if sec >= 3600 else "%d:%02d" % (sec // 60, sec % 60)


# ====================================================================== speakers
def _speaker_at(turns, starts, a, b):
    """Speaker whose turns overlap [a, b] the most; the nearest turn within 1 s if none overlaps."""
    if not turns:
        return None
    i = bisect.bisect_right(starts, b)
    best, best_ov = None, 0.0
    j = i - 1
    while j >= 0 and turns[j][0] > a - 600:          # turns are short; look back a bounded window
        s, e, spk = turns[j]
        ov = min(b, e) - max(a, s)
        if ov > best_ov:
            best, best_ov = spk, ov
        j -= 1
    if best is not None:
        return best
    mid, near, dist = (a + b) / 2, None, 1.0
    for k in (i - 1, i):
        if 0 <= k < len(turns):
            s, e, spk = turns[k]
            d = 0 if s <= mid <= e else min(abs(mid - s), abs(mid - e))
            if d <= dist:
                near, dist = spk, d
    return near


def assign_speakers(segments, turns, max_gap=2.0, max_len=45.0):
    """Turns whisper segments (with word timestamps) and diarization turns [(start, end, speaker)]
    into utterances [{start, end, speaker, text}]. Without turns, each segment is one utterance."""
    if not turns:
        return [{"start": s["start"], "end": s["end"], "speaker": None, "text": s["text"].strip()}
                for s in segments if s["text"].strip()]
    turns = sorted(turns)
    starts = [t[0] for t in turns]
    words = []
    for seg in segments:
        ws = seg.get("words") or [{"start": seg["start"], "end": seg["end"], "word": " " + seg["text"].strip()}]
        words.extend(w for w in ws if w.get("word", "").strip())
    out, prev = [], None
    for w in words:
        spk = _speaker_at(turns, starts, w["start"], w["end"]) or prev
        prev = spk
        cur = out[-1] if out else None
        if (cur and cur["speaker"] == spk and w["start"] - cur["end"] <= max_gap
                and not (w["end"] - cur["start"] > max_len and re.search(r"[.!?]$", cur["text"]))):
            cur["text"] += w["word"]
            cur["end"] = w["end"]
        else:
            out.append({"start": w["start"], "end": w["end"], "speaker": spk, "text": w["word"]})
    for u in out:
        u["text"] = re.sub(r"\s+", " ", u["text"]).strip()
    return [u for u in out if u["text"]]


def speaker_label(spk, names):
    if spk is None:
        return None
    return (names or {}).get(spk) or re.sub(r"^SPEAKER_0*(\d)", lambda m: "Speaker %d" % (int(m.group(1)) + 1), spk)


def speaker_stats(utterances):
    stats = {}
    for u in utterances:
        if u["speaker"] is None:
            continue
        s = stats.setdefault(u["speaker"], {"seconds": 0.0, "words": 0, "turns": 0})
        s["seconds"] += u["end"] - u["start"]
        s["words"] += len(u["text"].split())
        s["turns"] += 1
    for s in stats.values():
        s["seconds"] = round(s["seconds"], 1)
    return stats


# ====================================================================== formats
def cues(segments, utterances, max_chars=84, max_dur=6.0):
    """Subtitle cues built from words, never crossing a speaker change."""
    spk_of = []
    for u in utterances:
        spk_of.append((u["start"], u["end"], u["speaker"]))
    turns = sorted((s, e, k) for s, e, k in spk_of if k is not None)
    starts = [t[0] for t in turns]
    out = []
    for seg in segments:
        ws = seg.get("words") or [{"start": seg["start"], "end": seg["end"], "word": " " + seg["text"].strip()}]
        for w in ws:
            if not w.get("word", "").strip():
                continue
            spk = _speaker_at(turns, starts, w["start"], w["end"]) if turns else None
            cur = out[-1] if out else None
            if (cur and cur["speaker"] == spk and len(cur["text"]) + len(w["word"]) <= max_chars
                    and w["end"] - cur["start"] <= max_dur and w["start"] - cur["end"] < 1.0):
                cur["text"] += w["word"]
                cur["end"] = w["end"]
            else:
                out.append({"start": w["start"], "end": w["end"], "speaker": spk, "text": w["word"]})
    for c in out:
        c["text"] = c["text"].strip()
    return out


def render(fmt, data, names=None, title="Transcript"):
    """data: transcript.json content (segments, utterances, language, duration, summary...)."""
    utts = data.get("utterances") or []
    names = names or {}
    if fmt == "json":
        d = dict(data)
        d["speaker_names"] = names
        d["utterances"] = [dict(u, speaker_name=speaker_label(u["speaker"], names)) for u in utts]
        return json.dumps(d, indent=1, ensure_ascii=False)
    if fmt == "txt":
        lines = []
        for u in utts:
            who = speaker_label(u["speaker"], names)
            lines.append("[%s] %s%s" % (clock(u["start"]), who + ": " if who else "", u["text"]))
        return "\n".join(lines) + "\n"
    if fmt in ("srt", "vtt"):
        cs = cues(data.get("segments") or [], utts)
        if fmt == "srt":
            blocks = []
            for i, c in enumerate(cs, 1):
                who = speaker_label(c["speaker"], names)
                blocks.append("%d\n%s --> %s\n%s%s\n" % (i, ts(c["start"], ","), ts(c["end"], ","),
                                                         "[%s] " % who if who else "", c["text"]))
            return "\n".join(blocks)
        blocks = ["WEBVTT\n"]
        for c in cs:
            who = speaker_label(c["speaker"], names)
            blocks.append("%s --> %s\n%s%s\n" % (ts(c["start"]), ts(c["end"]), "<v %s>" % who if who else "", c["text"]))
        return "\n".join(blocks)
    if fmt == "md":
        out = ["# %s" % title, ""]
        meta = ["Duration %s" % clock(data.get("duration"))]
        if data.get("language"):
            meta.append("language `%s`" % data["language"])
        stats = speaker_stats(utts)
        if stats:
            meta.append("%d speakers" % len(stats))
        out += ["_" + " · ".join(meta) + "_", ""]
        if stats:
            total = sum(s["seconds"] for s in stats.values()) or 1
            out += ["| Speaker | Talk time | Share |", "|---|---|---|"]
            for spk, s in sorted(stats.items(), key=lambda kv: -kv[1]["seconds"]):
                out.append("| %s | %s | %d%% |" % (speaker_label(spk, names), clock(s["seconds"]),
                                                   round(100 * s["seconds"] / total)))
            out.append("")
        if data.get("summary"):
            out += [data["summary"].strip(), ""]
        out += ["## Transcript", ""]
        for u in utts:
            who = speaker_label(u["speaker"], names)
            out.append("**%s%s** %s\n" % ("[%s] " % clock(u["start"]), who + ":" if who else "", u["text"]))
        return "\n".join(out)
    raise ValueError("format must be one of: " + ", ".join(FORMATS))


# ====================================================================== summaries (Ollama)
SUMMARY_PROMPT = """You are a precise meeting assistant. Below is a transcript with timestamps and speaker names.
Write Markdown with exactly these sections:
## Summary
5-10 bullet points covering what was discussed.
## Decisions
Bullets; write "None recorded." if there are none.
## Action items
A table | Owner | Task | Due | using names from the transcript; "?" when unknown. "None recorded." if there are none.
## Open questions
Bullets; "None recorded." if there are none.
Only use facts from the transcript. Write in the transcript's language.{extra}

Transcript:
{text}"""

PART_PROMPT = """Write dense notes (bullets, keep names, numbers, decisions, tasks and owners) about this part
{n} of {total} of a meeting transcript. Only use facts from the text.{extra}

{text}"""


def ollama_chat(url, model, prompt, num_ctx=32768, timeout=1200):
    body = json.dumps({"model": model, "stream": False, "messages": [{"role": "user", "content": prompt}],
                       "options": {"num_ctx": num_ctx, "temperature": 0.2}}).encode()
    req = urllib.request.Request(url.rstrip("/") + "/api/chat", data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())["message"]["content"].strip()


def split_text(text, size):
    parts, cur = [], []
    n = 0
    for line in text.splitlines(keepends=True):
        if n + len(line) > size and cur:
            parts.append("".join(cur))
            cur, n = [], 0
        cur.append(line)
        n += len(line)
    if cur:
        parts.append("".join(cur))
    return parts


def summarize(text, url, model, instructions="", part_chars=60000, progress=None):
    """Map-reduce for long meetings: notes per part, then one summary from the notes."""
    extra = ("\nExtra instructions: " + instructions.strip()) if instructions and instructions.strip() else ""
    if len(text) <= part_chars:
        return ollama_chat(url, model, SUMMARY_PROMPT.format(text=text, extra=extra))
    parts = split_text(text, part_chars)
    notes = []
    for i, p in enumerate(parts, 1):
        if progress:
            progress("summarizing part %d/%d" % (i, len(parts)))
        notes.append("### Part %d\n%s" % (i, ollama_chat(url, model, PART_PROMPT.format(n=i, total=len(parts),
                                                                                     text=p, extra=extra))))
    if progress:
        progress("writing the final summary")
    return ollama_chat(url, model, SUMMARY_PROMPT.format(text="\n\n".join(notes), extra=extra))
''')
print("wrote", os.path.join(APP_DIR, 'whisper_core.py'))

In [ ]:
# Studio server on port 7860: jobs, API and MCP
# Writes whisper_server.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/whisper_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'whisper_server.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Whisper Diarization Studio: the meeting & audio OS on port 7860.

Runs as its own process (started by the notebook with studio_http.spawn):
    APP_PASSWORD=... HF_TOKEN=... python whisper_server.py --port 7860

Audio and video in (upload, URL, or a file under /kaggle/input) -> faster-whisper transcript ->
pyannote speaker diarization -> optional Ollama summary with decisions and action items ->
txt / md / srt / vtt / json. One job runs at a time; jobs live in WORK_DIR/jobs/<id>/ and survive
a server restart (an interrupted job is queued again).
"""
import argparse
import os
import re
import shutil
import subprocess
import threading
import time
import traceback
import uuid
import wave

import studio_http as K
import whisper_core as C

HERE = os.path.dirname(os.path.abspath(__file__))
WORK = os.environ.get("WORK_DIR", "/kaggle/working/whisper_studio")
JOBS_DIR = os.path.join(WORK, "jobs")
WHISPER_MODEL = os.environ.get("WHISPER_MODEL", "large-v3-turbo")
DIAR_MODEL = os.environ.get("DIARIZATION_MODEL", "pyannote/speaker-diarization-3.1")
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
SUMMARY_MODEL = os.environ.get("SUMMARY_MODEL", "")
HF_TOKEN = os.environ.get("HF_TOKEN") or None
INPUT_ROOTS = [r for r in ("/kaggle/input", "/kaggle/working") if os.path.isdir(r)] or [WORK]
MEDIA_EXT = (".mp3", ".wav", ".m4a", ".aac", ".flac", ".ogg", ".opus", ".wma", ".webm", ".mp4", ".mkv",
             ".mov", ".avi", ".m4v", ".amr", ".3gp")
MAX_UPLOAD = 4 << 30
MAX_WAIT = 85
log = K.file_logger(os.environ.get("APP_LOG", "/kaggle/working/logs/whisper.log"), "whisper")

LOCK = threading.RLock()
WAKE = threading.Condition(LOCK)
MODELS = {"whisper": None, "diar": None, "diar_error": None, "devices": None}
SUMMARIES = {}       # job id -> thread


# ====================================================================== job storage
def jdir(jid):
    jid = K.safe_name(jid, 40)
    d = os.path.join(JOBS_DIR, jid)
    if not jid or not os.path.isfile(os.path.join(d, "job.json")):
        raise K.HTTPError(404, "unknown job %r" % jid)
    return d


def load_job(jid):
    return K.read_json(os.path.join(jdir(jid), "job.json"))


def save_job(job, **changes):
    with LOCK:
        cur = K.read_json(os.path.join(JOBS_DIR, job["id"], "job.json"), job) or job
        cur.update(changes)
        cur["updated"] = time.time()
        K.write_json(os.path.join(JOBS_DIR, job["id"], "job.json"), cur)
        job.clear()
        job.update(cur)
    return job


def all_jobs():
    out = []
    if os.path.isdir(JOBS_DIR):
        for name in os.listdir(JOBS_DIR):
            j = K.read_json(os.path.join(JOBS_DIR, name, "job.json"))
            if j:
                out.append(j)
    return sorted(out, key=lambda j: -j["created"])


def public(job):
    keys = ("id", "title", "state", "step", "progress", "created", "updated", "error", "warnings", "duration",
            "language", "speakers", "options", "source", "summary_state", "summary_error", "seconds")
    return {k: job.get(k) for k in keys if k in job}


def parse_options(b):
    def opt_int(k, lo=1, hi=30):
        v = b.get(k)
        if v in (None, "", 0, "0"):
            return None
        v = int(v)
        if not lo <= v <= hi:
            raise ValueError("%s must be between %d and %d" % (k, lo, hi))
        return v

    def flag(k, default):
        v = b.get(k)
        if v in (None, ""):
            return default
        return v if isinstance(v, bool) else str(v).lower() in ("1", "true", "yes", "on")
    task = b.get("task") or "transcribe"
    if task not in ("transcribe", "translate"):
        raise ValueError("task must be 'transcribe' or 'translate' (to English)")
    lang = (b.get("language") or "").strip().lower() or None
    if lang and not re.fullmatch(r"[a-z]{2,3}", lang):
        raise ValueError("language must be a code like 'en', 'ne', 'hi' (or empty to detect)")
    return {"language": lang, "task": task, "diarize": flag("diarize", True),
            "num_speakers": opt_int("num_speakers"), "min_speakers": opt_int("min_speakers"),
            "max_speakers": opt_int("max_speakers"), "summarize": flag("summarize", bool(SUMMARY_MODEL)),
            "initial_prompt": str(b.get("initial_prompt") or "")[:800] or None, "vad": flag("vad", True)}


def new_job(source, options, title=None):
    jid = time.strftime("%Y%m%d-%H%M%S-") + uuid.uuid4().hex[:6]
    d = os.path.join(JOBS_DIR, jid)
    os.makedirs(d)
    job = {"id": jid, "title": (title or source.get("name") or source.get("value") or jid)[:120],
           "source": source, "options": options, "state": "new", "step": "receiving", "progress": 0.0,
           "created": time.time(), "warnings": []}
    K.write_json(os.path.join(d, "job.json"), job)
    return d, job


def enqueue(job):
    with WAKE:
        save_job(job, state="queued", step="waiting", progress=0.0, error=None)
        WAKE.notify_all()
    log.info("queued %s (%s)", job["id"], job["title"])
    return job


def safe_input_path(path):
    full = os.path.realpath(str(path or ""))
    if not any(full == r or full.startswith(os.path.realpath(r) + os.sep) for r in INPUT_ROOTS):
        raise ValueError("path must be under " + " or ".join(INPUT_ROOTS))
    if not os.path.isfile(full):
        raise ValueError("no such file: %s" % path)
    return full


def list_inputs():
    out = []
    for root in INPUT_ROOTS:
        for dp, dn, fn in os.walk(root):
            if dp.startswith(os.path.realpath(WORK)):
                dn[:] = []
                continue
            for f in sorted(fn):
                if f.lower().endswith(MEDIA_EXT):
                    p = os.path.join(dp, f)
                    out.append({"path": p, "size": os.path.getsize(p)})
                    if len(out) >= 300:
                        return out
    return out


# ====================================================================== models (loaded on first use)
def devices():
    if MODELS["devices"] is None:
        try:
            import torch
            n = torch.cuda.device_count()
        except Exception:
            n = 0
        # whisper on GPU 0; diarization on GPU 1 when there is one
        MODELS["devices"] = {"n": n, "whisper": 0 if n else None, "diar": (1 if n > 1 else 0) if n else None}
    return MODELS["devices"]


def whisper_model():
    if MODELS["whisper"] is None:
        from faster_whisper import WhisperModel
        dv = devices()
        if dv["whisper"] is None:
            MODELS["whisper"] = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")
        else:
            MODELS["whisper"] = WhisperModel(WHISPER_MODEL, device="cuda", device_index=dv["whisper"],
                                             compute_type="float16")
        log.info("loaded whisper %s", WHISPER_MODEL)
    return MODELS["whisper"]


def diarization_status():
    if not HF_TOKEN:
        return False, "add an HF_TOKEN secret (and accept the terms of %s and pyannote/segmentation-3.0)" % DIAR_MODEL
    if MODELS["diar_error"]:
        return False, MODELS["diar_error"]
    try:
        import importlib.util
        if importlib.util.find_spec("pyannote.audio") is None:
            return False, "pyannote.audio is not installed"
    except Exception:
        return False, "pyannote.audio is not installed"
    return True, "ready" if MODELS["diar"] else "loads on first use"


def diar_pipeline():
    if MODELS["diar"] is None:
        import torch
        from pyannote.audio import Pipeline
        try:
            pipe = Pipeline.from_pretrained(DIAR_MODEL, token=HF_TOKEN)              # pyannote.audio 4.x
        except TypeError:
            pipe = Pipeline.from_pretrained(DIAR_MODEL, use_auth_token=HF_TOKEN)     # pyannote.audio 3.x
        if pipe is None:
            raise RuntimeError("could not load %s: accept its terms (and pyannote/segmentation-3.0) on "
                               "huggingface.co with the account of your HF_TOKEN" % DIAR_MODEL)
        dv = devices()
        if dv["diar"] is not None:
            pipe.to(torch.device("cuda:%d" % dv["diar"]))
        MODELS["diar"] = pipe
        log.info("loaded diarization %s", DIAR_MODEL)
    return MODELS["diar"]


# ====================================================================== pipeline steps
def run(cmd, timeout=None):
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        raise RuntimeError("%s failed: %s" % (os.path.basename(cmd[0]), (p.stderr or p.stdout)[-1200:]))
    return p.stdout


def fetch_input(job, d):
    src = job["source"]
    if src["type"] in ("upload", "path"):
        return src["file"] if src["type"] == "upload" else safe_input_path(src["value"])
    save_job(job, step="downloading")
    url = src["value"]
    if shutil.which("yt-dlp"):
        run(["yt-dlp", "-q", "--no-playlist", "-f", "bestaudio/best", "--max-filesize", "4G",
             "-o", os.path.join(d, "input.%(ext)s"), url], timeout=3600)
        files = [f for f in os.listdir(d) if f.startswith("input.") and not f.endswith(".part")]
        if not files:
            raise RuntimeError("download produced no file")
        return os.path.join(d, files[0])
    import urllib.request
    path = os.path.join(d, "input" + (os.path.splitext(url.split("?")[0])[1][:6] or ".bin"))
    urllib.request.urlretrieve(url, path)
    return path


def convert(job, d, path):
    save_job(job, step="converting audio")
    wav = os.path.join(d, "audio.wav")
    run(["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y", "-i", path, "-vn", "-ac", "1",
         "-ar", "16000", "-c:a", "pcm_s16le", wav], timeout=3600)
    run(["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y", "-i", wav, "-c:a", "libmp3lame",
         "-b:a", "64k", os.path.join(d, "audio.mp3")], timeout=3600)
    with wave.open(wav) as w:
        duration = w.getnframes() / float(w.getframerate())
    if duration < 0.5:
        raise RuntimeError("the file has no usable audio")
    return wav, duration


def transcribe(job, wav, duration):
    o = job["options"]
    save_job(job, step="loading the speech model" if MODELS["whisper"] is None else "transcribing")
    model = whisper_model()
    save_job(job, step="transcribing")
    segs, info = model.transcribe(wav, language=o["language"], task=o["task"], word_timestamps=True,
                                  vad_filter=o["vad"], initial_prompt=o["initial_prompt"], beam_size=5)
    out, last = [], 0.0
    for s in segs:
        out.append({"start": round(s.start, 3), "end": round(s.end, 3), "text": s.text,
                    "words": [{"start": round(w.start, 3), "end": round(w.end, 3), "word": w.word,
                               "p": round(w.probability, 3)} for w in (s.words or [])]})
        if time.time() - last > 2:
            save_job(job, progress=round(min(0.99, s.end / duration), 3) * (0.75 if o["diarize"] else 1.0))
            last = time.time()
    return out, info.language


def diarize(job, wav):
    import numpy as np
    import torch
    o = job["options"]
    save_job(job, step="loading the diarization model" if MODELS["diar"] is None else "finding speakers",
             progress=0.75)
    pipe = diar_pipeline()
    save_job(job, step="finding speakers")
    with wave.open(wav) as w:
        audio = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16).astype(np.float32) / 32768.0
        sr = w.getframerate()
    kw = {k: o[k] for k in ("num_speakers", "min_speakers", "max_speakers") if o.get(k)}
    out = pipe({"waveform": torch.from_numpy(audio).unsqueeze(0), "sample_rate": sr}, **kw)
    ann = getattr(out, "speaker_diarization", out)       # 4.x returns DiarizeOutput, 3.x an Annotation
    return [(round(t.start, 3), round(t.end, 3), spk) for t, _, spk in ann.itertracks(yield_label=True)]


def process(job):
    d = os.path.join(JOBS_DIR, job["id"])
    t0 = time.time()
    save_job(job, state="running", progress=0.0, error=None, warnings=[])
    path = fetch_input(job, d)
    wav, duration = convert(job, d, path)
    save_job(job, duration=round(duration, 1))
    segments, language = transcribe(job, wav, duration)
    turns, warnings = [], []
    if job["options"]["diarize"]:
        ok, why = diarization_status()
        if ok:
            try:
                turns = diarize(job, wav)
            except Exception as e:
                log.error("diarization failed for %s\n%s", job["id"], traceback.format_exc())
                MODELS["diar_error"] = None if MODELS["diar"] else str(e)[:300]
                warnings.append("speaker diarization failed: %s" % str(e)[:300])
        else:
            warnings.append("no speaker labels: " + why)
    utterances = C.assign_speakers(segments, turns)
    data = {"language": language, "duration": round(duration, 1), "segments": segments, "utterances": utterances,
            "diarized": bool(turns), "whisper_model": WHISPER_MODEL, "options": job["options"]}
    K.write_json(os.path.join(d, "transcript.json"), data)
    if os.path.exists(wav) and os.path.exists(os.path.join(d, "audio.mp3")):
        os.remove(wav)                                  # keep the small mp3 for playback
    stats = C.speaker_stats(utterances)
    save_job(job, state="done", step="done", progress=1.0, language=language, warnings=warnings,
             speakers=sorted(stats), seconds=round(time.time() - t0, 1))
    log.info("done %s in %.0fs", job["id"], time.time() - t0)
    if job["options"]["summarize"] and SUMMARY_MODEL:
        start_summary(job["id"])


def worker():
    while True:
        with WAKE:
            queued = [j for j in all_jobs() if j["state"] == "queued"]
            if not queued:
                WAKE.wait(timeout=30)
                continue
            job = min(queued, key=lambda j: j["created"])
        try:
            process(job)
        except Exception as e:
            log.error("job %s failed\n%s", job["id"], traceback.format_exc())
            save_job(job, state="error", step="failed", error=str(e)[:1000])


# ====================================================================== summaries
def summary_text(jid, names=None):
    d = jdir(jid)
    data = K.read_json(os.path.join(d, "transcript.json"))
    if not data:
        raise ValueError("the transcript is not ready yet")
    return C.render("txt", data, names if names is not None else K.read_json(os.path.join(d, "speakers.json"), {}))


def start_summary(jid, instructions=""):
    if not SUMMARY_MODEL:
        raise ValueError("summaries are off: set SUMMARY_MODEL in the notebook's settings cell and re-run it")
    job = load_job(jid)
    if job["state"] != "done":
        raise ValueError("the transcript is not ready yet (state: %s)" % job["state"])
    t = SUMMARIES.get(jid)
    if t and t.is_alive():
        return job
    text = summary_text(jid)
    save_job(job, summary_state="running", summary_error=None, summary_step="starting")

    def go():
        try:
            md = C.summarize(text, OLLAMA_URL, SUMMARY_MODEL, instructions,
                             progress=lambda s: save_job(job, summary_step=s))
            with open(os.path.join(JOBS_DIR, jid, "summary.md"), "w", encoding="utf-8") as f:
                f.write(md + "\n")
            save_job(job, summary_state="done", summary_step="done")
        except Exception as e:
            log.error("summary %s failed\n%s", jid, traceback.format_exc())
            save_job(job, summary_state="error", summary_error=str(e)[:500])
    t = threading.Thread(target=go, daemon=True)
    SUMMARIES[jid] = t
    t.start()
    return job


def transcript_doc(jid, fmt):
    d = jdir(jid)
    job = load_job(jid)
    data = K.read_json(os.path.join(d, "transcript.json"))
    if not data:
        raise ValueError("the transcript is not ready yet (state: %s)" % job["state"])
    if os.path.exists(os.path.join(d, "summary.md")):
        data["summary"] = open(os.path.join(d, "summary.md"), encoding="utf-8").read()
    return C.render(fmt, data, K.read_json(os.path.join(d, "speakers.json"), {}), job["title"])


def rename_speakers(jid, names):
    if not isinstance(names, dict):
        raise ValueError("names must be an object like {\"SPEAKER_00\": \"Asha\"}")
    d = jdir(jid)
    cur = K.read_json(os.path.join(d, "speakers.json"), {})
    for k, v in names.items():
        v = str(v or "").strip()[:60]
        if v:
            cur[str(k)] = v
        else:
            cur.pop(str(k), None)
    K.write_json(os.path.join(d, "speakers.json"), cur)
    return cur


def wait_job(jid, seconds):
    seconds = max(0.0, min(MAX_WAIT, float(seconds or 0)))
    t = time.time()
    while True:
        job = load_job(jid)
        busy = job["state"] in ("queued", "running") or job.get("summary_state") == "running"
        if not busy or time.time() - t >= seconds:
            return job
        time.sleep(1.5)


def delete_job(jid):
    job = load_job(jid)
    if job["state"] == "running":
        raise ValueError("this job is running; wait for it to finish")
    shutil.rmtree(jdir(jid))
    return {"deleted": jid}


def info():
    ok, why = diarization_status()
    jobs = all_jobs()
    return {"whisper_model": WHISPER_MODEL, "diarization": {"available": ok, "detail": why, "model": DIAR_MODEL},
            "summary_model": SUMMARY_MODEL or None, "gpus": K.gpu_info(),
            "queue": {s: sum(1 for j in jobs if j["state"] == s) for s in ("queued", "running", "done", "error")}}


# ====================================================================== app
def build_app():
    app = K.App("Whisper Diarization Studio", password=os.environ.get("APP_PASSWORD"), log=log,
                max_body=MAX_UPLOAD, instructions=(
                    "Transcribes audio/video with speaker labels and meeting summaries. transcribe() queues a job "
                    "from a URL or a file path under /kaggle/input (see list_input_files) and returns a job_id; poll "
                    "get_job (wait_seconds up to 85) until state is 'done', then read get_transcript (format md "
                    "includes the summary). Rename speakers with rename_speakers. Uploads go through REST: "
                    "POST /api/jobs/upload?name=<file> with the raw file body."))
    app.page("/", os.path.join(HERE, "whisper_ui.html"))
    app.static("/static/", HERE)

    @app.route("GET", "/api/info")
    def _info(req):
        return info()

    @app.route("GET", "/api/jobs")
    def _jobs(req):
        return {"jobs": [public(j) for j in all_jobs()]}

    @app.route("POST", "/api/jobs/upload")
    def _upload(req):
        name = K.safe_name(req.arg("name", "upload.bin"), 100) or "upload.bin"
        if not name.lower().endswith(MEDIA_EXT):
            raise ValueError("unsupported file type; send audio or video (%s)" % " ".join(MEDIA_EXT))
        opts = parse_options(req.query)
        d, job = new_job({"type": "upload", "name": name}, opts, req.arg("title"))
        try:
            path = os.path.join(d, "input_" + name)
            size = req.save_body(path, MAX_UPLOAD)
        except BaseException:
            shutil.rmtree(d, ignore_errors=True)
            raise
        save_job(job, source=dict(job["source"], file=path, size=size))
        return public(enqueue(job))

    @app.route("POST", "/api/jobs")
    def _create(req):
        b = req.json()
        return public(create_from(b))

    @app.route("GET", r"/api/jobs/(?P<jid>[\w-]+)")
    def _job(req):
        jid = req.params["jid"]
        job = wait_job(jid, req.arg("wait", 0, float))
        d = jdir(jid)
        out = public(job)
        data = K.read_json(os.path.join(d, "transcript.json"))
        if data:
            out["utterances"] = data["utterances"]
            out["diarized"] = data.get("diarized")
            out["speaker_stats"] = C.speaker_stats(data["utterances"])
        out["speaker_names"] = K.read_json(os.path.join(d, "speakers.json"), {})
        if os.path.exists(os.path.join(d, "summary.md")):
            out["summary"] = open(os.path.join(d, "summary.md"), encoding="utf-8").read()
        out["has_audio"] = os.path.exists(os.path.join(d, "audio.mp3"))
        return out

    @app.route("GET", r"/api/jobs/(?P<jid>[\w-]+)/download/(?P<fmt>\w+)")
    def _download(req):
        fmt = req.params["fmt"]
        if fmt not in C.FORMATS:
            raise ValueError("format must be one of " + ", ".join(C.FORMATS))
        job = load_job(req.params["jid"])
        name = "%s.%s" % (K.safe_name(job["title"], 60) or job["id"], fmt)
        return K.Response(transcript_doc(job["id"], fmt), 200, C.FORMATS[fmt] + "; charset=utf-8",
                          {"Content-Disposition": K._disposition(name)})

    @app.route("GET", r"/api/jobs/(?P<jid>[\w-]+)/audio")
    def _audio(req):
        return K.FileResponse(os.path.join(jdir(req.params["jid"]), "audio.mp3"), ctype="audio/mpeg")

    @app.route("POST", r"/api/jobs/(?P<jid>[\w-]+)/speakers")
    def _speakers(req):
        return {"speaker_names": rename_speakers(req.params["jid"], req.json().get("names"))}

    @app.route("POST", r"/api/jobs/(?P<jid>[\w-]+)/summarize")
    def _summarize(req):
        return public(start_summary(req.params["jid"], str(req.json().get("instructions") or "")[:1000]))

    @app.route("POST", r"/api/jobs/(?P<jid>[\w-]+)/retry")
    def _retry(req):
        job = load_job(req.params["jid"])
        if job["state"] in ("queued", "running"):
            raise ValueError("job is already %s" % job["state"])
        return public(enqueue(job))

    @app.route("POST", r"/api/jobs/(?P<jid>[\w-]+)/delete")
    def _delete(req):
        return delete_job(req.params["jid"])

    @app.route("GET", "/api/sources")
    def _sources(req):
        return {"files": list_inputs()}

    # ---------------------------------------------------------------- MCP tools
    opts_schema = {
        "title": {"type": "string"},
        "language": {"type": "string", "description": "e.g. 'en', 'ne', 'hi'; omit to detect"},
        "task": {"type": "string", "enum": ["transcribe", "translate"], "description": "translate = into English"},
        "diarize": {"type": "boolean", "default": True},
        "num_speakers": {"type": "integer", "description": "exact number of speakers, if known"},
        "min_speakers": {"type": "integer"}, "max_speakers": {"type": "integer"},
        "summarize": {"type": "boolean", "description": "default: on when a summary model is configured"},
        "initial_prompt": {"type": "string", "description": "names and jargon to help spelling"},
    }

    @app.tool("transcribe", "Queue a transcription job from a URL (direct file or any yt-dlp site) or a file path "
              "under /kaggle/input. Returns the job; poll get_job until state is 'done'.",
              dict({"url": {"type": "string"}, "path": {"type": "string"}}, **opts_schema))
    def t_transcribe(url=None, path=None, **opts):
        return public(create_from(dict(opts, url=url, path=path)))

    @app.tool("get_job", "Job status, progress and speakers. wait_seconds (max 85) waits for it to finish.", {
        "job_id": {"type": "string"}, "wait_seconds": {"type": "number", "default": 0}}, ["job_id"])
    def t_get_job(job_id, wait_seconds=0):
        return public(wait_job(job_id, wait_seconds))

    @app.tool("get_transcript", "The transcript as txt (timestamps + speakers), md (with summary and talk time), "
              "srt, vtt or json. Long transcripts are paged: pass offset=next_offset.", {
                  "job_id": {"type": "string"}, "format": {"type": "string", "enum": list(C.FORMATS), "default": "md"},
                  "offset": {"type": "integer", "default": 0}, "max_chars": {"type": "integer", "default": 60000}},
              ["job_id"])
    def t_transcript(job_id, format="md", offset=0, max_chars=60000):
        if format not in C.FORMATS:
            raise ValueError("format must be one of " + ", ".join(C.FORMATS))
        text = transcript_doc(job_id, format)
        offset, max_chars = max(0, int(offset)), max(1000, min(200000, int(max_chars)))
        part = text[offset:offset + max_chars]
        nxt = offset + len(part)
        return part + ("\n\n[... %d more characters: call again with offset=%d]" % (len(text) - nxt, nxt)
                       if nxt < len(text) else "")

    @app.tool("list_jobs", "Recent jobs, newest first.", {"limit": {"type": "integer", "default": 20}})
    def t_list(limit=20):
        return [public(j) for j in all_jobs()[:max(1, int(limit))]]

    @app.tool("rename_speakers", "Give speakers real names, e.g. {\"SPEAKER_00\": \"Asha\"}. An empty name "
              "resets one.", {"job_id": {"type": "string"}, "names": {"type": "object"}}, ["job_id", "names"])
    def t_rename(job_id, names):
        return {"speaker_names": rename_speakers(job_id, names)}

    @app.tool("summarize", "(Re)write the meeting summary: key points, decisions, action items, open questions. "
              "Uses the current speaker names.", {
                  "job_id": {"type": "string"}, "instructions": {"type": "string"},
                  "wait_seconds": {"type": "number", "default": 60}}, ["job_id"])
    def t_summarize(job_id, instructions="", wait_seconds=60):
        start_summary(job_id, str(instructions or "")[:1000])
        job = wait_job(job_id, wait_seconds)
        if job.get("summary_state") == "done":
            return open(os.path.join(jdir(job_id), "summary.md"), encoding="utf-8").read()
        return public(job)

    @app.tool("delete_job", "Delete a job and its files.", {"job_id": {"type": "string"}}, ["job_id"])
    def t_delete(job_id):
        return delete_job(job_id)

    @app.tool("list_input_files", "Audio and video files attached to the notebook (under /kaggle/input).")
    def t_inputs():
        return list_inputs()

    @app.tool("server_status", "Models, diarization availability, GPU memory and queue counts.")
    def t_status():
        return info()

    return app


def create_from(b):
    url, path = (b.get("url") or "").strip(), (b.get("path") or "").strip()
    if bool(url) == bool(path):
        raise ValueError("give exactly one of url or path")
    opts = parse_options(b)
    if url:
        if not re.match(r"^https?://", url):
            raise ValueError("url must start with http:// or https://")
        source = {"type": "url", "value": url}
    else:
        source = {"type": "path", "value": safe_input_path(path), "name": os.path.basename(path)}
    d, job = new_job(source, opts, b.get("title"))
    return enqueue(job)


def resume_interrupted():
    for j in all_jobs():
        if j["state"] == "running":
            save_job(j, state="queued", step="waiting (restarted)")
        if j.get("summary_state") == "running":
            save_job(j, summary_state="error", summary_error="interrupted by a restart; run it again")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--port", type=int, default=7860)
    os.makedirs(JOBS_DIR, exist_ok=True)
    resume_interrupted()
    threading.Thread(target=worker, daemon=True).start()
    build_app().serve_forever(ap.parse_args().port)
''')
print("wrote", os.path.join(APP_DIR, 'whisper_server.py'))

In [ ]:
# Studio page
# Writes whisper_ui.html into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/whisper_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'whisper_ui.html'), "w", encoding="utf-8") as f:
    f.write(r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Whisper Studio</title>
<link rel="stylesheet" href="/static/kit.css">
<style>
.layout{display:grid;grid-template-columns:330px 1fr;gap:16px;align-items:start}
@media (max-width:900px){.layout{grid-template-columns:1fr}}
.joblist{display:flex;flex-direction:column;gap:6px;max-height:70vh;overflow:auto}
.jobitem{border:1px solid var(--line);border-radius:8px;padding:8px 10px;cursor:pointer;background:var(--panel2)}
.jobitem.on{border-color:var(--accent)}
.jobitem .t{font-weight:600;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.utt{display:grid;grid-template-columns:62px 130px 1fr;gap:8px;padding:6px 4px;border-bottom:1px solid var(--line);cursor:pointer}
.utt:hover{background:var(--panel2)}
.utt.now{background:color-mix(in srgb,var(--accent) 12%,transparent)}
.utt .who{font-weight:600}
.transcript{max-height:60vh;overflow:auto}
.summary{white-space:pre-wrap;background:var(--panel2);border-radius:8px;padding:12px;font-size:13.5px}
.spk input{width:140px}
audio{width:100%}
@media (max-width:640px){.utt{grid-template-columns:52px 1fr}.utt .who{grid-column:2}.utt .tx{grid-column:1/3}}
</style></head><body>
<header class="top">
  <h1>Whisper Diarization Studio</h1><span class="sub" id="models"></span>
  <div class="right small muted" id="status"></div>
</header>
<main>
<div class="card">
  <h2>New transcription</h2>
  <div class="grid">
    <div class="col">
      <div class="drop" id="drop">Drop audio or video here, or click to choose<br><span class="small">Up to 100 MB through Cloudflare; attach bigger files as a Kaggle dataset.</span></div>
      <input type="file" id="file" accept="audio/*,video/*" multiple hidden>
      <div class="row"><input id="url" placeholder="…or a URL (direct file, YouTube, podcast page)" style="flex:1"><button id="addUrl">Add URL</button></div>
      <div class="row"><select id="path" style="flex:1"><option value="">…or a file attached to the notebook</option></select><button id="addPath">Add file</button></div>
    </div>
    <div class="col">
      <div class="row">
        <label>Language<input id="language" placeholder="auto" style="width:80px"></label>
        <label>Task<select id="task"><option value="transcribe">Transcribe</option><option value="translate">Translate to English</option></select></label>
        <label>Speakers<input id="num_speakers" type="number" min="1" max="30" placeholder="auto" style="width:80px"></label>
      </div>
      <div class="row">
        <label class="inline"><input type="checkbox" id="diarize" checked> Who spoke when</label>
        <label class="inline"><input type="checkbox" id="summarize" checked> Meeting summary</label>
      </div>
      <label>Names and terms to spell right (optional)<input id="initial_prompt" placeholder="e.g. Gunjan, Kathmandu, Kubernetes"></label>
      <div class="small muted" id="diarNote"></div>
    </div>
  </div>
</div>
<div class="layout">
  <div class="card"><h2>Jobs</h2><div class="joblist" id="jobs"><div class="empty">No jobs yet.</div></div></div>
  <div class="card" id="detail"><div class="empty">Pick a job to see its transcript.</div></div>
</div>
<div class="card"><h2>For agents</h2><p class="small muted">MCP tools: transcribe, get_job, get_transcript, list_jobs, rename_speakers, summarize, delete_job, list_input_files, server_status. Uploads: <code>POST /api/jobs/upload?name=file.mp3</code> with the raw file as the body.</p><pre class="log" id="mcp"></pre></div>
</main>
<script src="/static/kit.js"></script>
<script>
$("#mcp").textContent = mcpHelp("whisper");
let current = null, currentData = null, info = null;

function opts() {
  const o = {language: $("#language").value.trim(), task: $("#task").value, diarize: $("#diarize").checked,
             summarize: $("#summarize").checked, initial_prompt: $("#initial_prompt").value.trim()};
  if ($("#num_speakers").value) o.num_speakers = +$("#num_speakers").value;
  return o;
}

async function upload(files) {
  for (const f of files) {
    const q = new URLSearchParams({name: f.name, ...Object.fromEntries(Object.entries(opts()).map(([k, v]) => [k, String(v)]))});
    toast("Uploading " + f.name + " (" + fmtBytes(f.size) + ")…");
    const j = await guard(() => api("/api/jobs/upload?" + q, {body: f}), "Queued " + f.name);
    current = j.id;
  }
  loadJobs();
}
dropZone($("#drop"), $("#file"), upload);
$("#addUrl").onclick = async () => { const url = $("#url").value.trim(); if (!url) return;
  const j = await guard(() => api("/api/jobs", {json: {url, ...opts()}}), "Queued"); $("#url").value = ""; current = j.id; loadJobs(); };
$("#addPath").onclick = async () => { const path = $("#path").value; if (!path) return;
  const j = await guard(() => api("/api/jobs", {json: {path, ...opts()}}), "Queued"); current = j.id; loadJobs(); };

async function loadInfo() {
  info = await api("/api/info");
  $("#models").textContent = "whisper " + info.whisper_model + (info.summary_model ? " · summaries " + info.summary_model : "");
  $("#status").textContent = info.queue.running + " running · " + info.queue.queued + " queued";
  $("#diarNote").textContent = info.diarization.available ? "" : "Speaker labels are off: " + info.diarization.detail;
  if (!info.summary_model) { $("#summarize").checked = false; $("#summarize").disabled = true; }
}
async function loadSources() {
  const {files} = await api("/api/sources");
  for (const f of files) $("#path").append(el("option", {value: f.path}, f.path.replace("/kaggle/input/", "") + " (" + fmtBytes(f.size) + ")"));
}

async function loadJobs() {
  const {jobs} = await api("/api/jobs");
  const box = $("#jobs");
  box.replaceChildren(...jobs.map(j => el("div", {class: "jobitem" + (j.id === current ? " on" : ""), onclick: () => { current = j.id; loadJobs(); showJob(); }},
    el("div", {class: "t"}, j.title),
    el("div", {class: "row small muted"}, badge(j.state), j.state === "running" ? j.step + " " + Math.round(j.progress * 100) + "%" : fmtAgo(j.created),
       j.duration ? "· " + fmtDur(j.duration) : ""),
    j.state === "running" ? progressBar(j.progress) : null)));
  if (!jobs.length) box.append(el("div", {class: "empty"}, "No jobs yet."));
  if (!current && jobs.length) { current = jobs[0].id; showJob(); }
  const cur = jobs.find(j => j.id === current);
  if (cur && currentData && (cur.state !== currentData.state || cur.summary_state !== currentData.summary_state || cur.state === "running")) showJob();
}

function label(spk) { if (!spk) return ""; return (currentData.speaker_names || {})[spk] || spk.replace(/^SPEAKER_0*(\d+)/, (m, n) => "Speaker " + (+n + 1)); }

async function showJob() {
  if (!current) return;
  const j = currentData = await api("/api/jobs/" + current);
  const d = $("#detail");
  const head = el("div", {class: "row", style: {marginBottom: "10px"}}, el("h2", {style: {margin: 0}}, j.title), badge(j.state),
    j.duration ? el("span", {class: "muted small"}, fmtDur(j.duration) + (j.language ? " · " + j.language : "")) : null,
    el("span", {style: {marginLeft: "auto"}, class: "row"},
      j.state === "error" ? el("button", {onclick: () => guard(() => api("/api/jobs/" + j.id + "/retry", {json: {}}), "Queued again").then(loadJobs)}, "Retry") : null,
      el("button", {class: "danger", onclick: async () => { if (!confirm("Delete this job and its files?")) return;
        await guard(() => api("/api/jobs/" + j.id + "/delete", {json: {}}), "Deleted"); current = null; d.replaceChildren(el("div", {class: "empty"}, "Deleted.")); loadJobs(); }}, "Delete")));
  const parts = [head];
  if (j.state === "queued" || j.state === "running") parts.push(el("div", {class: "col"}, el("div", {}, j.step), progressBar(j.progress)));
  if (j.error) parts.push(el("pre", {class: "log"}, j.error));
  (j.warnings || []).forEach(w => parts.push(el("div", {class: "small muted"}, "Note: " + w)));
  if (j.state === "done") {
    const audio = j.has_audio ? el("audio", {controls: true, src: "/api/jobs/" + j.id + "/audio", preload: "metadata"}) : null;
    if (audio) parts.push(audio);
    parts.push(el("div", {class: "row", style: {margin: "10px 0"}}, "Download:",
      ...["md", "txt", "srt", "vtt", "json"].map(f => el("a", {class: "btn", href: "/api/jobs/" + j.id + "/download/" + f}, f))));
    const stats = j.speaker_stats || {};
    if (Object.keys(stats).length) {
      const rows = Object.entries(stats).sort((a, b) => b[1].seconds - a[1].seconds).map(([spk, s]) =>
        el("div", {class: "row spk"}, el("span", {class: "mono", style: {width: "100px"}}, spk),
           el("input", {value: (j.speaker_names || {})[spk] || "", placeholder: label(spk), "data-spk": spk}),
           el("span", {class: "small muted"}, fmtDur(s.seconds) + " · " + s.words + " words")));
      parts.push(el("h3", {}, "Speakers"), ...rows, el("button", {onclick: async () => {
        const names = {}; $$("input[data-spk]", d).forEach(i => names[i.dataset.spk] = i.value.trim());
        await guard(() => api("/api/jobs/" + j.id + "/speakers", {json: {names}}), "Names saved"); showJob(); }}, "Save names"));
    }
    if (info && info.summary_model) {
      const sumBtn = el("button", {onclick: async () => { await guard(() => api("/api/jobs/" + j.id + "/summarize", {json: {instructions: $("#sumx").value}}), "Summarizing…"); showJob(); }},
        j.summary ? "Rewrite summary" : "Write summary");
      parts.push(el("h3", {}, "Summary"), el("div", {class: "row"}, el("input", {id: "sumx", placeholder: "Extra instructions (optional)", style: {flex: 1}}), sumBtn));
      if (j.summary_state === "running") parts.push(el("div", {class: "small muted"}, "Writing the summary… (uses the current speaker names)"));
      if (j.summary_state === "error") parts.push(el("div", {class: "small", style: {color: "var(--bad)"}}, j.summary_error));
      if (j.summary) parts.push(el("div", {class: "summary", style: {marginTop: "8px"}}, j.summary));
    }
    const tr = el("div", {class: "transcript"}, ...(j.utterances || []).map(u => el("div", {class: "utt", "data-s": u.start, "data-e": u.end,
      onclick: () => { if (audio) { audio.currentTime = u.start; audio.play(); } }},
      el("span", {class: "mono muted"}, fmtDur(u.start)), el("span", {class: "who"}, label(u.speaker)), el("span", {class: "tx"}, u.text))));
    parts.push(el("h3", {}, "Transcript"), tr);
    if (audio) audio.addEventListener("timeupdate", () => {
      const t = audio.currentTime;
      $$(".utt", tr).forEach(x => x.classList.toggle("now", t >= +x.dataset.s && t < +x.dataset.e));
    });
  }
  d.replaceChildren(...parts);
}

loadSources().catch(() => {});
every(5000, loadInfo);
every(3000, loadJobs);
</script></body></html>
''')
print("wrote", os.path.join(APP_DIR, 'whisper_ui.html'))

In [ ]:
# 3. Load the helpers
import os, sys
APP_DIR = '/kaggle/working/whisper_app'
_missing = [f for f in ("studio_http.py", "kit.css", "kit.js") if not os.path.exists(os.path.join(APP_DIR, f))]
if _missing:
    raise RuntimeError("Run the 'Writes ... into APP_DIR' cells above first (missing: %s)" % ", ".join(_missing))
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
import studio_http as K   # Notebook helpers: kaggle_secret, spawn, wait_http, tail, start_tunnel, keep_alive

In [ ]:
# 4. Install (quiet; about 4 minutes; re-running skips finished steps)
import shutil, subprocess
def pip_install(*args, keep_core=True):
    """pip install, quietly. keep_core pins Kaggle's own torch / numpy builds so nothing replaces them."""
    cons = "/tmp/keep_core.txt"
    lines = []
    if keep_core:
        import importlib.metadata as md
        for pkg in ("torch", "torchvision", "torchaudio", "numpy"):
            try:
                lines.append("%s==%s" % (pkg, md.version(pkg)))
            except md.PackageNotFoundError:
                pass
    open(cons, "w").write("\n".join(lines) + "\n")
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-c", cons, *args]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-2000:], p.stderr[-3000:])
        raise RuntimeError("pip install failed: " + " ".join(args))

os.makedirs(LOG_DIR, exist_ok=True)
if not shutil.which("ffmpeg"):
    print("ffmpeg...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, capture_output=True)
print("faster-whisper, yt-dlp...")
pip_install("faster-whisper", "yt-dlp", "nvidia-cublas-cu12", "nvidia-cudnn-cu12")

DIARIZATION_OK = True
print("pyannote.audio (speaker diarization)...")
try:
    pip_install("pyannote.audio")
except RuntimeError as e:
    DIARIZATION_OK = False
    print("  pyannote.audio did not install next to this Kaggle image's PyTorch; transcripts will have no speakers.")

if SUMMARY_MODEL and not shutil.which("ollama"):
    print("Ollama...")
    subprocess.run("apt-get -qq install -y zstd && curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, capture_output=True)
K.ensure_cloudflared()

print("Downloading the Whisper model %s ..." % WHISPER_MODEL)
from faster_whisper import download_model
download_model(WHISPER_MODEL)
import torch
print("Install complete. GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "none (CPU is very slow)")

In [ ]:
# 5. Start Ollama for summaries (GPU 1 when there are two) and pull the model. Log: /kaggle/working/logs/ollama.log
import torch
N_GPU = torch.cuda.device_count()
if SUMMARY_MODEL:
    ollama_env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", CUDA_VISIBLE_DEVICES="1" if N_GPU > 1 else "0",
                      OLLAMA_CONTEXT_LENGTH="32768", OLLAMA_KEEP_ALIVE="30m")
    K.spawn("ollama", ["ollama", "serve"], LOG_DIR + "/ollama.log", env=ollama_env)
    if not K.wait_http("http://127.0.0.1:11434/api/tags", timeout=60, name="ollama"):
        print(K.tail(LOG_DIR + "/ollama.log"))
        raise RuntimeError("Ollama did not start; log above. Or set SUMMARY_MODEL = \"\" to skip summaries.")
    print("Pulling %s (first time ~5 GB)..." % SUMMARY_MODEL)
    p = subprocess.run(["ollama", "pull", SUMMARY_MODEL], env=ollama_env, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stderr[-1500:])
        raise RuntimeError("ollama pull failed")
    print("Summary model ready.")
else:
    K.stop_process("ollama")
    print("Summaries are off (SUMMARY_MODEL is empty).")

In [ ]:
# 6. Start the studio on port 7860 (background process). Log: /kaggle/working/logs/whisper.log
import glob, site
# CTranslate2 (faster-whisper) loads cuBLAS / cuDNN from the pip-installed NVIDIA wheels
LIB_DIRS = sorted({os.path.dirname(p) for sp in site.getsitepackages() + [site.getusersitepackages()]
                   for pat in ("nvidia/cublas/lib/*.so*", "nvidia/cudnn/lib/*.so*") for p in glob.glob(os.path.join(sp, pat))})
PASSWORD = K.ui_password("WHISPER_UI_PASSWORD")
HF_TOKEN = K.kaggle_secret("HF_TOKEN")
if not HF_TOKEN:
    print("No HF_TOKEN secret: transcripts will not have speaker labels (see the first cell).")
env = dict(os.environ, APP_PASSWORD=PASSWORD, WORK_DIR=WORK_DIR, APP_LOG=LOG_DIR + "/whisper.log",
           WHISPER_MODEL=WHISPER_MODEL, DIARIZATION_MODEL=DIARIZATION_MODEL,
           SUMMARY_MODEL=SUMMARY_MODEL, OLLAMA_URL="http://127.0.0.1:11434", PYTHONUNBUFFERED="1",
           LD_LIBRARY_PATH=":".join(LIB_DIRS + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(":"))
env.pop("HF_TOKEN", None)
if HF_TOKEN and globals().get("DIARIZATION_OK", True):
    env["HF_TOKEN"] = HF_TOKEN
K.spawn("whisper-studio", [sys.executable, os.path.join(APP_DIR, "whisper_server.py"), "--port", str(PORT)],
        LOG_DIR + "/whisper.log", env=env, cwd=APP_DIR)
if not K.wait_http("http://127.0.0.1:%d/health" % PORT, timeout=60, name="whisper-studio"):
    print(K.tail(LOG_DIR + "/whisper.log", 40))
    raise RuntimeError("The studio did not start; log above")
for host in ("127.0.0.1", "[::1]"):
    print(host, "->", "up" if K.wait_http("http://%s:%d/health" % (host, PORT), timeout=3) else "not reachable")
print(K.api_get(PORT, "/api/info", PASSWORD)["diarization"])

In [ ]:
# 7. Publish through your Cloudflare Tunnel (token from the WHISPER_TUNNEL_TOKEN secret, never printed)
tunnel = K.start_tunnel("WHISPER_TUNNEL_TOKEN", PORT, LOG_DIR + "/cloudflared.log")

In [ ]:
# 8. Keep-alive monitor. Leave it running; interrupting it only stops the status lines.
def queue_line():
    q = K.api_get(PORT, "/api/info", PASSWORD)["queue"]
    return "jobs: %d running, %d queued, %d done" % (q["running"], q["queued"], q["done"])

K.keep_alive(PORT, ["whisper-studio", "ollama", "cloudflared"] if SUMMARY_MODEL else ["whisper-studio", "cloudflared"],
             extra=queue_line)